# Word2Vec from Scratch — C to Python Walkthrough

This notebook walks through **word2vec.c** (Mikolov et al., 2013) section by section: we explain each part and reimplement it in Python so you can run and inspect every step.

**Reference:** [Efficient Estimation of Word Representations in Vector Space](https://arxiv.org/abs/1301.3781)  
**Original C code:** [tmikolov/word2vec](https://github.com/tmikolov/word2vec)

---
## 1. Constants, types, and global state (C lines 1–49)

**What the C code does:**
- **Includes:** `stdio`, `stdlib`, `string`, `math`, `pthread` for I/O, memory, strings, math, and multi-threading.
- **Constants:**
  - `MAX_STRING`: max characters per word (100).
  - `EXP_TABLE_SIZE` / `MAX_EXP`: precomputed sigmoid table to avoid calling `exp()` in the inner loop.
  - `MAX_SENTENCE_LENGTH`: max words per sentence (1000).
  - `MAX_CODE_LENGTH`: max depth of the Huffman tree (40).
  - `vocab_hash_size`: size of the hash table for vocabulary lookup (30M).
- **`struct vocab_word`:** For each word we store: count (`cn`), Huffman code and path (`code`, `point`, `codelen`), and the word string.
- **Global variables:** File paths, vocab array, hash table, hyperparameters (`window`, `min_count`, `cbow`, `hs`, `negative`, etc.), and the weight matrices `syn0` (input/embedding), `syn1` (output for hierarchical softmax), `syn1neg` (output for negative sampling).

**Python equivalent:** We use a small set of constants and a config object; we avoid true globals by passing state through classes or function arguments.

In [2]:
import numpy as np
from io import StringIO
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# === Constants (mirroring C #defines and const) ===
MAX_STRING = 100
EXP_TABLE_SIZE = 1000
MAX_EXP = 6.0
MAX_SENTENCE_LENGTH = 1000
MAX_CODE_LENGTH = 40
VOCAB_HASH_SIZE = 30_000_000  # max ~21M words with 0.7 load factor
TABLE_SIZE = int(1e8)  # for negative sampling unigram table


@dataclass
class VocabWord:
    """One entry in the vocabulary (C: struct vocab_word)."""
    word: str
    cn: int  # count
    code: List[int] = field(default_factory=list)   # Huffman code (0/1)
    point: List[int] = field(default_factory=list) # path from root to leaf
    codelen: int = 0


@dataclass
class Config:
    """Hyperparameters and paths (C: global variables)."""
    train_file: str = ""
    output_file: str = ""
    layer1_size: int = 100
    window: int = 5
    min_count: int = 5
    cbow: int = 1
    hs: int = 0
    negative: int = 5
    iter: int = 5
    alpha: float = 0.025
    sample: float = 1e-3
    num_threads: int = 1  # we keep 1 for clarity; can parallelize later
    binary: int = 0
    debug_mode: int = 2


print("Constants and types defined. VocabWord = word + count + Huffman code/point.")

Constants and types defined. VocabWord = word + count + Huffman code/point.


---
## 2. Precomputed exp table (C: main, before TrainModel)

**What the C code does:** In the inner loop we need σ(x) = 1/(1+e^{-x}). Computing `exp()` every time is slow. So the C code precomputes a table for values in [-MAX_EXP, MAX_EXP] and maps x to an index: `expTable[(int)((f + MAX_EXP) * (EXP_TABLE_SIZE / MAX_EXP / 2))]`. The stored value is actually **f(x) = exp(x) / (exp(x)+1)** (i.e. sigmoid).

**Python:** Build the same table once and use it during training.

In [3]:
def build_exp_table(exp_table_size: int = EXP_TABLE_SIZE, max_exp: float = MAX_EXP) -> np.ndarray:
    """Precompute sigmoid(f) for f in [-max_exp, max_exp] (C: expTable in main)."""
    exp_table = np.zeros(exp_table_size + 1, dtype=np.float32)
    for i in range(exp_table_size):
        # Map index i to value in [-max_exp, max_exp]
        f = (i / exp_table_size * 2 - 1) * max_exp
        exp_table[i] = np.exp(f) / (np.exp(f) + 1)
    return exp_table


def sigmoid_from_table(f: float, exp_table: np.ndarray) -> float:
    """Look up sigmoid(f), clipping to table range (C: same index computation)."""
    if f <= -MAX_EXP:
        return 0.0
    if f >= MAX_EXP:
        return 1.0
    idx = int((f + MAX_EXP) * (EXP_TABLE_SIZE / MAX_EXP / 2))
    idx = min(idx, EXP_TABLE_SIZE - 1)
    return exp_table[idx]


exp_table = build_exp_table()
print("exp_table shape:", exp_table.shape)
print("sigmoid(-1) ≈", sigmoid_from_table(-1.0, exp_table), "(expected ≈ 0.27)")

exp_table shape: (1001,)
sigmoid(-1) ≈ 0.26737145 (expected ≈ 0.27)


---
## 3. Read one word from file (C: ReadWord)

**What the C code does:** Reads characters until a space, tab, or newline. Treats newline as a special token `</s>` (sentence boundary). Truncates words longer than `MAX_STRING-1`. Sets `eof` when EOF is reached.

**Python:** Same logic using a file handle or an iterator over tokens; we use a simple tokenizer that yields words and `</s>`.

In [ ]:
def read_word(stream) -> Tuple[str, bool]:
    """
    Read one word from stream. Word = chars until space/tab/newline.
    Newline is returned as '</s>'. Returns (word, eof).
    (C: ReadWord(char *word, FILE *fin, char *eof))
    """
    buf = []
    while True:
        ch = stream.read(1)
        if not ch:
            return ("".join(buf), True)
        if ch == "\r":
            continue
        if ch in " \t\n":
            if buf:
                if ch == "\n":
                    stream.seek(stream.tell() - 1)  # ungetc
                return ("".join(buf), False)
            if ch == "\n":
                return ("</s>", False)
            continue
        buf.append(ch)
        if len(buf) >= MAX_STRING - 1:
            buf = buf[: MAX_STRING - 1]
    return ("".join(buf), False)


# Quick test
s = StringIO("hello world\n</s>\nfoo bar")
w, eof = read_word(s); print(repr(w), eof)
w, eof = read_word(s); print(repr(w), eof)
w, eof = read_word(s); print(repr(w), eof)
w, eof = read_word(s); print(repr(w), eof)

---
## 4. Hash and vocabulary lookup (C: GetWordHash, SearchVocab)

**What the C code does:**
- **GetWordHash:** Computes a hash of the word string (polynomial hash with base 257) and takes it modulo `vocab_hash_size`.
- **SearchVocab:** Uses linear probing: starts at `vocab_hash[hash]`; if the slot is -1 the word is not in the vocab; otherwise compares the string; if not equal, advances to the next slot until found or -1.

**Python:** We use a dict for the hash table (word → index) for simplicity; the C logic is equivalent to a hash table with linear probing.

In [ ]:
def get_word_hash(word: str, vocab_hash_size: int = VOCAB_HASH_SIZE) -> int:
    """Polynomial hash (C: GetWordHash)."""
    h = 0
    for c in word:
        h = (h * 257 + ord(c)) % (2 ** 64)
    return h % vocab_hash_size


class Vocabulary:
    """
    Vocabulary: list of VocabWord + hash table for O(1) lookup.
    (C: vocab[] + vocab_hash[])
    """
    def __init__(self, vocab_hash_size: int = VOCAB_HASH_SIZE):
        self.vocab: List[VocabWord] = []
        self.vocab_hash_size = vocab_hash_size
        self.vocab_hash: List[int] = [-1] * vocab_hash_size  # -1 = empty
        self.train_words = 0

    def search(self, word: str) -> int:
        """Return index of word in vocab, or -1 (C: SearchVocab)."""
        h = get_word_hash(word, self.vocab_hash_size)
        while True:
            idx = self.vocab_hash[h]
            if idx == -1:
                return -1
            if self.vocab[idx].word == word:
                return idx
            h = (h + 1) % self.vocab_hash_size
        return -1

    def add(self, word: str) -> int:
        """Add word to vocab, return its index (C: AddWordToVocab)."""
        idx = len(self.vocab)
        self.vocab.append(VocabWord(word=word, cn=0))
        h = get_word_hash(word, self.vocab_hash_size)
        while self.vocab_hash[h] != -1:
            h = (h + 1) % self.vocab_hash_size
        self.vocab_hash[h] = idx
        return idx


voc = Vocabulary(vocab_hash_size=1000)  # small for demo
voc.add("</s>")
voc.add("hello")
voc.add("world")
print("search 'hello':", voc.search("hello"))
print("search 'missing':", voc.search("missing"))

---
## 5. Sort vocabulary and prune rare words (C: SortVocab)

**What the C code does:**
1. Sorts the vocabulary by **decreasing count** (except index 0 which is `</s>`).
2. Rebuilds the hash table (indices changed after sort).
3. **Prunes** words with count < `min_count` (except `</s>`).
4. Allocates `code` and `point` arrays for each word (for the Huffman tree later).

**Python:** Same: sort by count descending, rebuild hash, drop rare words, reserve code/point.

In [8]:
def sort_vocab(voc: Vocabulary, min_count: int) -> None:
    """
    Sort by count (desc), rebuild hash, remove words with cn < min_count.
    (C: SortVocab)
    """
    # Keep </s> at index 0, sort the rest by count descending
    rest = voc.vocab[1:]
    rest.sort(key=lambda w: -w.cn)
    voc.vocab = [voc.vocab[0]] + rest
    # Rebuild hash and prune
    voc.vocab_hash = [-1] * voc.vocab_hash_size
    voc.train_words = 0
    new_vocab = []
    for a, w in enumerate(voc.vocab):
        if w.cn < min_count and a != 0:
            continue
        new_vocab.append(w)
        h = get_word_hash(w.word, voc.vocab_hash_size)
        while voc.vocab_hash[h] != -1:
            h = (h + 1) % voc.vocab_hash_size
        voc.vocab_hash[h] = len(new_vocab) - 1
        voc.train_words += w.cn
    voc.vocab = new_vocab
    for w in voc.vocab:
        w.code = [0] * MAX_CODE_LENGTH
        w.point = [0] * MAX_CODE_LENGTH


# Demo: add words with counts, then sort and prune
v = Vocabulary(1000)
v.add("</s>"); v.vocab[0].cn = 100
for w, c in [("the", 50), ("a", 3), ("cat", 20)]:
    i = v.add(w)
    v.vocab[i].cn = c
sort_vocab(v, min_count=5)
print("After sort_vocab(min_count=5):", [(w.word, w.cn) for w in v.vocab])

NameError: name 'Vocabulary' is not defined

---
## 6. Unigram table for negative sampling (C: InitUnigramTable)

**What the C code does:** Negative sampling draws “negative” words (non-context words) from a distribution. The paper uses the **unigram distribution raised to power 0.75** (so frequent words are sampled more often, but not proportionally). The C code builds a large table `table[0..table_size-1]` where each index maps to a word index; the number of times word i appears in the table is proportional to count(i)^0.75. Sampling is then: pick a random index in the table and use that word.

**Python:** Build a list of word indices with multiplicity ~ count^0.75 (normalized).

In [ ]:
def init_unigram_table(vocab: List[VocabWord], table_size: int = TABLE_SIZE, power: float = 0.75) -> np.ndarray:
    """
    Build table for negative sampling: table[i] = word index.
    Frequency of word j in table ∝ vocab[j].cn^power.
    (C: InitUnigramTable)
    """
    vocab_size = len(vocab)
    train_words_pow = sum(vocab[a].cn ** power for a in range(vocab_size))
    table = np.zeros(table_size, dtype=np.int64)
    d1 = (vocab[0].cn ** power) / train_words_pow
    i = 0
    for a in range(table_size):
        table[a] = i
        if a / table_size > d1:
            i = min(i + 1, vocab_size - 1)
            d1 += (vocab[i].cn ** power) / train_words_pow
    return table


vocab_demo = [VocabWord("</s>", 10), VocabWord("a", 100), VocabWord("b", 50)]
t = init_unigram_table(vocab_demo, table_size=1000)
print("Unigram table (first 20):", t[:20])
print("Count of 0:", np.sum(t == 0), "Count of 1:", np.sum(t == 1), "Count of 2:", np.sum(t == 2))

---
## 7. Huffman tree for hierarchical softmax (C: CreateBinaryTree)

**What the C code does:** Instead of a full softmax over V words (O(V)), hierarchical softmax uses a binary tree: each word is a leaf. The path from root to leaf is a binary code; internal nodes have vectors. Probability of the word = product of binary decisions along the path. **Frequent words get shorter codes** (fewer multiplications). The C code:
1. Builds a Huffman tree from word counts (two smallest nodes merged repeatedly).
2. For each word, stores the path (point = indices of internal nodes) and the code (0/1 at each step).

**Python:** Same algorithm: build tree, then for each leaf record code and point.

In [ ]:
def create_binary_tree(vocab: List[VocabWord]) -> None:
    """
    Build Huffman tree from word counts; fill vocab[i].code and vocab[i].point.
    (C: CreateBinaryTree)
    """
    vocab_size = len(vocab)
    count = np.full(vocab_size * 2 + 1, 1e15, dtype=np.float64)
    count[:vocab_size] = [vocab[a].cn for a in range(vocab_size)]
    binary = np.zeros(vocab_size * 2 + 1, dtype=np.int64)
    parent_node = np.zeros(vocab_size * 2 + 1, dtype=np.int64)

    pos1 = vocab_size - 1
    pos2 = vocab_size
    for a in range(vocab_size - 1):
        if pos1 >= 0:
            if count[pos1] < count[pos2]:
                min1i, pos1 = pos1, pos1 - 1
            else:
                min1i, pos2 = pos2, pos2 + 1
        else:
            min1i, pos2 = pos2, pos2 + 1
        if pos1 >= 0:
            if count[pos1] < count[pos2]:
                min2i, pos1 = pos1, pos1 - 1
            else:
                min2i, pos2 = pos2, pos2 + 1
        else:
            min2i, pos2 = pos2, pos2 + 1
        count[vocab_size + a] = count[min1i] + count[min2i]
        parent_node[min1i] = vocab_size + a
        parent_node[min2i] = vocab_size + a
        binary[min2i] = 1

    for a in range(vocab_size):
        b = a
        code_list, point_list = [], []
        while True:
            code_list.append(binary[b])
            point_list.append(b)
            b = parent_node[b]
            if b == vocab_size * 2 - 2:
                break
        vocab[a].codelen = len(code_list)
        vocab[a].point[0] = vocab_size - 2
        for b in range(len(code_list)):
            vocab[a].code[len(code_list) - b - 1] = code_list[b]
            vocab[a].point[len(code_list) - b] = point_list[b] - vocab_size


v2 = [VocabWord("</s>", 10), VocabWord("a", 5), VocabWord("b", 3), VocabWord("c", 2)]
for w in v2:
    w.code = [0] * MAX_CODE_LENGTH
    w.point = [0] * MAX_CODE_LENGTH
create_binary_tree(v2)
for i, w in enumerate(v2):
    print(w.word, "codelen", w.codelen, "code", w.code[: w.codelen], "point", w.point[: w.codelen + 1])

---
## 8. Learn vocabulary from training file (C: LearnVocabFromTrainFile)

**What the C code does:** Opens the training file, adds `</s>` as first word, then for each word read: increment total word count, search vocab; if not found add it with count 1, else increment its count. If vocab size exceeds 70% of hash size, call **ReduceVocab** (discard words with count <= min_reduce). Finally call **SortVocab** and optionally print stats.

**Python:** Same: stream over file, add/increment counts, optional reduce, then sort_vocab.

In [ ]:
def learn_vocab_from_train_file(
    train_path: str,
    voc: Vocabulary,
    min_count: int,
    vocab_hash_size: int = VOCAB_HASH_SIZE,
    debug: bool = True,
) -> int:
    """
    Build vocabulary from training file. (C: LearnVocabFromTrainFile)
    Returns file_size (bytes) for later seeking in multi-threaded training.
    """
    voc.vocab_hash = [-1] * voc.vocab_hash_size
    voc.vocab = []
    voc.train_words = 0
    voc.add("</s>")
    min_reduce = 1
    with open(train_path, "r", encoding="utf-8", errors="ignore") as f:
        content = f.read()
    stream = StringIO(content)
    while True:
        word, eof = read_word(stream)
        if eof:
            break
        voc.train_words += 1
        i = voc.search(word)
        if i == -1:
            a = voc.add(word)
            voc.vocab[a].cn = 1
        else:
            voc.vocab[i].cn += 1
        if len(voc.vocab) > vocab_hash_size * 0.7:
            # ReduceVocab: drop words with cn <= min_reduce
            voc.vocab = [w for w in voc.vocab if w.cn > min_reduce]
            voc.vocab_hash = [-1] * voc.vocab_hash_size
            for a, w in enumerate(voc.vocab):
                h = get_word_hash(w.word, voc.vocab_hash_size)
                while voc.vocab_hash[h] != -1:
                    h = (h + 1) % voc.vocab_hash_size
                voc.vocab_hash[h] = a
            min_reduce += 1
    sort_vocab(voc, min_count)
    if debug:
        print("Vocab size:", len(voc.vocab), "Train words:", voc.train_words)
    return len(content)

---
## 9. Initialize network (C: InitNet)

**What the C code does:**
- **syn0:** embedding matrix, shape (vocab_size, layer1_size). Initialized with uniform random in [-0.5/layer1_size, 0.5/layer1_size] using a fixed LCG (linear congruential generator) seed.
- **syn1:** if hierarchical softmax (hs=1), output weights for internal nodes, shape (vocab_size, layer1_size), init to 0.
- **syn1neg:** if negative sampling (negative>0), output weights for negative sampling, same shape, init to 0.
- Then calls **CreateBinaryTree** to fill code/point for each word.

**Python:** Allocate NumPy arrays, same init; then create_binary_tree.

In [ ]:
def lcg_next(next_random: int) -> int:
    """C uses: next_random = next_random * 25214903917 + 11"""
    return (next_random * 25214903917 + 11) & 0xFFFFFFFF_FFFFFFFF


def init_net(
    vocab: List[VocabWord],
    layer1_size: int,
    hs: int,
    negative: int,
    seed: int = 1,
) -> Tuple[np.ndarray, Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Allocate and initialize syn0, syn1 (if hs), syn1neg (if negative).
    (C: InitNet)
    """
    vocab_size = len(vocab)
    next_random = seed
    syn0 = np.zeros((vocab_size, layer1_size), dtype=np.float32)
    for a in range(vocab_size):
        for b in range(layer1_size):
            next_random = lcg_next(next_random)
            syn0[a, b] = ((next_random & 0xFFFF) / 65536.0 - 0.5) / layer1_size
    syn1 = None
    if hs:
        syn1 = np.zeros((vocab_size, layer1_size), dtype=np.float32)
    syn1neg = None
    if negative > 0:
        syn1neg = np.zeros((vocab_size, layer1_size), dtype=np.float32)
    create_binary_tree(vocab)
    return syn0, syn1, syn1neg


voc3 = [VocabWord("</s>", 5), VocabWord("a", 3), VocabWord("b", 2)]
for w in voc3:
    w.code = [0] * MAX_CODE_LENGTH
    w.point = [0] * MAX_CODE_LENGTH
s0, s1, s1n = init_net(voc3, layer1_size=4, hs=0, negative=2)
print("syn0 shape:", s0.shape)
print("syn0 sample:", s0[0])

---
## 10. Training one step: CBOW + Skip-gram with HS and negative sampling

**What the C code does (TrainModelThread):**
- For each sentence position, get center word `word` and context words in window.
- **CBOW:** neu1 = average of context word embeddings (syn0). Predict `word` from neu1 (HS or negative sampling). Backprop gradient into neu1e, then add neu1e to all context word rows of syn0.
- **Skip-gram:** For each context word, use center word embedding (syn0[word]) and predict that context word (HS or negative sampling). Update syn0[word] and output weights.
- **Subsampling:** With probability depending on count, skip frequent words (formula in C line ~296).
- **Learning rate:** Linear decay: alpha = starting_alpha * (1 - progress).

**Python:** One function that processes a list of sentences (list of word indices), implements one pass over the data with the same update rules.

In [ ]:
def train_epoch(
    sentences: List[List[int]],  # list of sentences, each = list of word indices
    syn0: np.ndarray,
    syn1: Optional[np.ndarray],
    syn1neg: Optional[np.ndarray],
    vocab: List[VocabWord],
    exp_table: np.ndarray,
    unigram_table: Optional[np.ndarray],
    config: Config,
) -> None:
    """
    One epoch of training. (C: one thread of TrainModelThread, simplified.)
    We only implement negative sampling here (no HS) for brevity.
    """
    layer1_size = config.layer1_size
    window = config.window
    alpha = config.alpha
    negative = config.negative
    cbow = config.cbow
    vocab_size = len(vocab)
    train_words = sum(w.cn for w in vocab)
    sample = config.sample

    next_random = 1
    for sen in sentences:
        for sentence_position, word in enumerate(sen):
            if word < 0:
                continue
            # Subsampling (C: sample > 0 block)
            if sample > 0 and vocab[word].cn > 0:
                ran = (np.sqrt(vocab[word].cn / (sample * train_words)) + 1) * (sample * train_words) / vocab[word].cn
                next_random = lcg_next(next_random)
                if ran < (next_random & 0xFFFF) / 65536.0:
                    continue
            b = next_random % window if window else 0
            neu1 = np.zeros(layer1_size, dtype=np.float32)
            neu1e = np.zeros(layer1_size, dtype=np.float32)

            if cbow:
                cw = 0
                for a in range(2 * window + 1):
                    if a == window:
                        continue
                    c = sentence_position - window + a
                    if c < 0 or c >= len(sen):
                        continue
                    last_word = sen[c]
                    if last_word < 0:
                        continue
                    neu1 += syn0[last_word]
                    cw += 1
                if cw == 0:
                    continue
                neu1 /= cw
                # Negative sampling
                if negative > 0 and syn1neg is not None and unigram_table is not None:
                    for d in range(negative + 1):
                        if d == 0:
                            target, label = word, 1
                        else:
                            next_random = lcg_next(next_random)
                            target = int(unigram_table[(next_random >> 16) % len(unigram_table)])
                            if target == 0:
                                target = next_random % (vocab_size - 1) + 1
                            if target == word:
                                continue
                            label = 0
                        f = float(np.dot(neu1, syn1neg[target]))
                        if f > MAX_EXP:
                            g = (label - 1) * alpha
                        elif f < -MAX_EXP:
                            g = (label - 0) * alpha
                        else:
                            g = (label - sigmoid_from_table(f, exp_table)) * alpha
                        neu1e += g * syn1neg[target]
                        syn1neg[target] += g * neu1
                for a in range(2 * window + 1):
                    if a == window:
                        continue
                    c = sentence_position - window + a
                    if c < 0 or c >= len(sen):
                        continue
                    last_word = sen[c]
                    if last_word >= 0:
                        syn0[last_word] += neu1e
            else:
                # Skip-gram
                for a in range(2 * window + 1):
                    if a == window:
                        continue
                    c = sentence_position - window + a
                    if c < 0 or c >= len(sen):
                        continue
                    last_word = sen[c]
                    if last_word < 0:
                        continue
                    l1 = last_word
                    neu1e = np.zeros(layer1_size, dtype=np.float32)
                    if negative > 0 and syn1neg is not None and unigram_table is not None:
                        for d in range(negative + 1):
                            if d == 0:
                                target, label = word, 1
                            else:
                                next_random = lcg_next(next_random)
                                target = int(unigram_table[(next_random >> 16) % len(unigram_table)])
                                if target == 0:
                                    target = next_random % (vocab_size - 1) + 1
                                if target == word:
                                    continue
                                label = 0
                            f = float(np.dot(syn0[l1], syn1neg[target]))
                            if f > MAX_EXP:
                                g = (label - 1) * alpha
                            elif f < -MAX_EXP:
                                g = (label - 0) * alpha
                            else:
                                g = (label - sigmoid_from_table(f, exp_table)) * alpha
                            neu1e += g * syn1neg[target]
                            syn1neg[target] += g * syn0[l1]
                    syn0[l1] += neu1e


print("train_epoch (CBOW/Skip-gram + negative sampling) defined.")

---
## 11. End-to-end: build vocab, init net, train, save

**What the C code does (TrainModel, main):**
1. Parse CLI args (or use defaults).
2. LearnVocabFromTrainFile (or ReadVocab).
3. Optionally SaveVocab.
4. InitNet; if negative > 0 then InitUnigramTable.
5. Build expTable.
6. Spawn threads, each running TrainModelThread (different file offset).
7. Join threads; write syn0 to output file (or run K-means if classes > 0).

**Python:** We run a single-threaded pipeline on a small in-memory corpus and write vectors to a text file.

In [ ]:
def sentences_to_indices(sentences: List[List[str]], voc: Vocabulary) -> List[List[int]]:
    """Convert list of sentences (words) to list of sentence index lists. OOV = -1."""
    out = []
    for sen in sentences:
        idxs = []
        for w in sen:
            i = voc.search(w)
            idxs.append(i if i >= 0 else -1)
        out.append(idxs)
    return out


def train_word2vec(
    train_path: str,
    output_path: str,
    config: Optional[Config] = None,
) -> Tuple[Vocabulary, np.ndarray]:
    """Full pipeline: learn vocab, init net, train, save vectors."""
    config = config or Config()
    if config.cbow:
        config.alpha = 0.05
    voc = Vocabulary()
    learn_vocab_from_train_file(train_path, voc, config.min_count, debug=True)
    vocab = voc.vocab
    syn0, syn1, syn1neg = init_net(
        vocab, config.layer1_size, config.hs, config.negative
    )
    unigram_table = None
    if config.negative > 0:
        unigram_table = init_unigram_table(vocab, table_size=min(TABLE_SIZE, len(vocab) * 1000))
    exp_tab = build_exp_table()

    # Build sentences (word indices) from file again
    with open(train_path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    stream = StringIO(text)
    sentences = []
    current = []
    while True:
        w, eof = read_word(stream)
        if eof:
            if current:
                sentences.append(current)
            break
        if w == "</s>":
            if current:
                sentences.append(current)
            current = []
            continue
        i = voc.search(w)
        current.append(i if i >= 0 else -1)
    if current:
        sentences.append(current)

    # Train
    for epoch in range(config.iter):
        train_epoch(sentences, syn0, syn1, syn1neg, vocab, exp_tab, unigram_table, config)
        print("Epoch", epoch + 1, "done.")

    if output_path:
        with open(output_path, "w", encoding="utf-8") as fo:
            fo.write(f"{len(vocab)} {config.layer1_size}\n")
            for a in range(len(vocab)):
                fo.write(vocab[a].word + " " + " ".join(map(str, syn0[a])) + "\n")
        print("Vectors saved to", output_path)
    return voc, syn0

---
## 12. Demo: tiny corpus

Run on a small text file to verify the pipeline and inspect embeddings.

In [ ]:
demo_corpus = """the cat sat on the mat .
the dog sat on the mat .
cats and dogs are animals .
the cat and the dog played .
"""
with open("demo_train.txt", "w") as f:
    f.write(demo_corpus)

cfg = Config(
    layer1_size=8,
    window=2,
    min_count=1,
    cbow=0,  # skip-gram
    negative=3,
    iter=5,
    alpha=0.025,
    sample=0,
)
voc, syn0 = train_word2vec("demo_train.txt", "demo_vectors.txt", config=cfg)

print("\nEmbedding for 'cat':", syn0[voc.search("cat")])
print("Embedding for 'dog':", syn0[voc.search("dog")])
print("Cosine sim(cat, dog):", np.dot(syn0[voc.search("cat")], syn0[voc.search("dog")]) / (
    np.linalg.norm(syn0[voc.search("cat")]) * np.linalg.norm(syn0[voc.search("dog")]) + 1e-9
))

---
## 13. Using libraries — when and which

**When to use the from-scratch code:** Learning how Word2Vec works, debugging, or matching the C reference line-by-line.

**When to use libraries:** Real projects: faster training (C/optimized), less code, maintained APIs, and extra features (phrases, loading/saving, similarity, analogies).

| Library | Pros | Use case |
|--------|------|----------|
| **gensim** | Same algorithm as word2vec.c, simple API, `Word2Vec`, `KeyedVectors` | Training and loading Word2Vec models in Python |
| **PyTorch / TensorFlow** | Custom training, GPU, integrate with other layers | Research or custom embedding pipelines |
| **sentence-transformers** | Sentence (not word) embeddings, pre-trained | Semantic similarity, retrieval |

Below: the same tiny corpus with **gensim** so you can compare outputs (and speed on larger data). Install: `pip install gensim`.

In [5]:
%pip install gensim

  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 0.8/24.4 MB 10.1 MB/s eta 0:00:03
   --- ------------------------------------ 1.8/24.4 MB 5.2 MB/s eta 0:00:05
   ----- ---------------------------------- 3.1/24.4 MB 6.7 MB/s eta 0:00:04
   ------ --------------------------------- 3.7/24.4 MB 5.0 MB/s eta 0:00:05
   -------- ------------------------------- 5.2/24.4 MB 5.8 MB/s eta 0:00:04
   ----------- ---------------------------- 6.8/24.4 MB 5.7 MB/s eta 0:00:04
   ------------ --------------------------- 7.3/24.4 MB 5.9 MB/s eta 0:00:03
   --------------- ------------------------ 9.7/24.4 MB 6.0 MB/s eta 0:00:03
   ------------------ --------------------- 11.5/24.4 MB 6.6 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.4 MB 6.6 MB/s eta 0:00:02
   ---------------------- ----------------- 13.6/24.4 MB 6.3 MB/s eta 0:00:02
   ----------

In [7]:
# Same demo with gensim (optional: pip install gensim)
try:
    from gensim.models import Word2Vec

    # Same corpus as the from-scratch demo (no file needed — works in any working dir)
    sentences = [
        ["the", "cat", "sat", "on", "the", "mat", "."],
        ["the", "dog", "sat", "on", "the", "mat", "."],
        ["cats", "and", "dogs", "are", "animals", "."],
        ["the", "cat", "and", "the", "dog", "played", "."],
    ]
    model = Word2Vec(
        sentences,
        vector_size=8,
        window=2,
        min_count=1,
        sg=1,  # 1 = skip-gram, 0 = CBOW
        negative=3,
        epochs=5,
        alpha=0.025,
        seed=42,
    )

    wv = model.wv
    print("Gensim vocab size:", len(wv))
    if "cat" in wv and "dog" in wv:
        print("Embedding for 'cat':", wv["cat"])
        print("Embedding for 'dog':", wv["dog"])
        print("Cosine sim(cat, dog):", wv.similarity("cat", "dog"))
        print("Most similar to 'cat':", wv.most_similar("cat", topn=3))
except ImportError:
    print("Install gensim: pip install gensim")

Gensim vocab size: 13
Embedding for 'cat': [ 0.01162244 -0.08644642  0.06088038  0.04574066  0.10562278  0.06119116
 -0.0333339   0.11687731]
Embedding for 'dog': [ 0.07035177  0.03590651 -0.02433807  0.08067782  0.0114286  -0.0141572
 -0.01240761 -0.06811686]
Cosine sim(cat, dog): -0.25828966
Most similar to 'cat': [('.', 0.6867055296897888), ('and', 0.6677999496459961), ('mat', 0.40038394927978516)]
